In [ ]:
GEMINI_API_KEY =
GEMINI_MODEL = "gemini-3.5-flash"

In [26]:
import os
import re
import json
import time
from pathlib import Path
from typing import Optional

from google import genai
from google.genai import types

VIDEO_EXTENSIONS = {".mp4"}

complete = []

def wait_file_active(client, uploaded_file, timeout_sec: int = 300):
    start = time.time()

    while True:
        f = client.files.get(name=uploaded_file.name)

        state_name = getattr(getattr(f, "state", None), "name", None)

        if state_name == "ACTIVE":
            return f

        if state_name == "FAILED":
            raise RuntimeError(f"Gemini не смог обработать файл: {uploaded_file.name}")

        if time.time() - start > timeout_sec:
            raise TimeoutError(f"Файл слишком долго обрабатывается: {uploaded_file.name}")

        print(f"Gemini обрабатывает файл: {uploaded_file.name}")
        time.sleep(5)

def extract_json(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    cleaned = text.strip()

    cleaned = re.sub(r"^```json\s*", "", cleaned)
    cleaned = re.sub(r"^```\s*", "", cleaned)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"raw_response": text}


def find_profile_dir(number_dir: Path) -> Optional[Path]:
    if not number_dir.exists() or not number_dir.is_dir():
        return None

    subdirs = [p for p in number_dir.iterdir() if p.is_dir()]

    # Идеальный случай: внутри папки 1/2/3/... одна папка с id
    if len(subdirs) == 1:
        candidate = subdirs[0]
        if (candidate / "profile.json").exists() or (candidate / "bio.txt").exists():
            return candidate

    # Запасной вариант: если вдруг там несколько папок, берём ту, где есть profile.json
    for candidate in subdirs:
        if (candidate / "profile.json").exists():
            return candidate

    return None


def get_video_paths(profile_dir: Path, limit: int = 3) -> list[Path]:
    videos_dir = profile_dir / "videos"

    if not videos_dir.exists():
        return []

    videos = [
        p for p in videos_dir.iterdir()
        if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS
    ]

    videos.sort()

    return videos[:limit]


def read_profile_files(profile_dir: Path) -> tuple[str, dict]:
    bio_path = profile_dir / "bio.txt"
    profile_json_path = profile_dir / "profile.json"

    bio_text = ""
    profile_data = {}

    if bio_path.exists():
        bio_text = bio_path.read_text(encoding="utf-8", errors="ignore")

    if profile_json_path.exists():
        with open(profile_json_path, "r", encoding="utf-8") as f:
            profile_data = json.load(f)

    return bio_text, profile_data


def build_prompt(profile_dir: Path, bio_text: str, profile_data: dict) -> str:
    return f"""Роль: Ты — ИИ-экстрактор семантических и мультимодальных признаков (Multimodal Feature Extractor) для платформы инфлюенс-маркетинга. Твоя задача — преобразовать сырые текстовые данные профиля и прикрепленные видеофайлы в плотный, машинно-читаемый JSON-объект, оптимизированный исключительно для генерации векторных эмбеддингов (text embeddings). 

Входные данные: 
1. Bio профиля: 
<bio_text> 
{bio_text}
</bio_text> 
2. Метаданные профиля и массив постов: 
<profile_json> 
{json.dumps(profile_data, ensure_ascii=False, indent=2)}
</profile_json> 
3. Три (3) прикрепленных видеофайла. 

Инструкции: 
- КРИТИЧЕСКОЕ СИНТАКСИЧЕСКОЕ ПРАВИЛО: Не пиши полные предложения, объяснения или повествования. Выводи ТОЛЬКО отдельные ключевые слова, technical-токены и короткие фразы (максимум 1-3 слова), разделенные запятыми внутри каждого значения JSON. Полностью исключи стоп-слова, предлоги, союзы и грамматические связки. 

- ПРАВИЛО РАСШИРЕНИЯ СИНОНИМАМИ: Ты должен обогатить извлеченные данные, добавив 5-10 связанных индустриальных терминов, синонимов и поисковых тегов. Это расширение применяется ко ВСЕМ полям JSON, КРОМЕ полей "entities", "brands" и "blogger_gender". Запрещено создавать новые ключи JSON. Все синонимы должны быть просто словами через запятую внутри предопределенных ключей. 

- ПРАВИЛО ГЛУБОКОЙ ДЕТАЛИЗАЦИИ ПРОДУКТОВ: При извлечении типов, категорий или наименований продуктов (таких как крем, сыворотка, маска, скраб, масло, патчи и т.д.) ты обязан на основе контекста текста или видеоряда конкретизировать их функциональное назначение или зону применения. Не оставляй продукт обобщенным. Если это применимо и следует из контекста, раскрывай его структуру (например: вместо "сыворотка" извлекай "сыворотка для лица", вместо "масло" — "масло для кончиков волос", вместо "крем" — "крем для кожи вокруг глаз"). Не выдумывай абстрактные примеры, используй только ту анатомическую или целевую привязку, которая явно демонстрируется в кадре или упоминается в материалах.

- СТРОГОЕ ПРАВИЛО ДЛЯ "ENTITIES" (ТОЛЬКО ГЕОГРАФИЯ И ИМЕНА НА ЯЗЫКЕ ОРИГИНАЛА): Поле "entities" предназначено строго для фактических данных локаций и персон. Оно должно содержать ТОЛЬКО явно упомянутые или четко видимые географические объекты, города, страны и имена людей из текста или видео (например: "СПБ", "USA", "Москва", "Милан"). КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО включать сюда названия коммерческих брендов, продуктов, компаний и интернет-платформ (они уходят в поле "brands"). НЕ добавляй в это поле синонимы, догадки или обобщения.


- СТРОГОЕ ПРАВИЛО ДЛЯ "BRANDS" (ТОЛЬКО КОММЕРЧЕСКИЕ БРЕНДЫ И ПЛАТФОРМЫ НА ЯЗЫКЕ ОРИГИНАЛА): Поле "brands" предназначено строго для фиксации коммерческого контекста. Сюда должны попадать только конкретные торговые марки, бренды, а также платформы дистрибуции (маркетплейсы и магазины).
  * ПРИОРИТЕТ №1 (Бьюти-индустрия): В первую очередь ищи и извлекай бьюти-бренды (косметика, уход, парфюмерия, бьюти-гаджеты), так как выборка блогеров сформирована под задачи рекламы бьюти-продуктов (например: "Celimax", "Shik Studio", "Mixit", "d'Alba", "Dyson").
  * Смежные категории: Также обязательно фиксируй бренды из категорий: одежда, украшения/аксессуары, бытовая химия и любые близкие по смыслу потребительские товары или услуги (например: "Zara", "Swarovski", "Synergetic", "Grass").
  * Торговые платформы и маркетплейсы: Обязательно извлекай названия площадок и магазинов, на которых заказываются или упоминаются товары (например: "Wildberries", "Ozon", "Золотое Яблоко", "Лэтуаль").
  * КАТЕГОРИЧЕСКИ ЗАПРЕЩЕНО: Добавлять в это поле города, страны, локации, имена людей или обобщенные категории (должно быть название бренда/магазина, а не слова "косметика", "магазин", "одежда"). 
  * Языковые ограничения: Все сущности ДОЛЖНЫ БЫТЬ НА ЯЗЫКЕ ОРИГИНАЛА (в их официальном коммерческом написании, без транслитерации и перевода). НЕ добавляй сюда синонимы или догадки.

- ПРАВИЛО ОПРЕДЕЛЕНИЯ ПОЛА БЛОГЕРА: В поле "blogger_gender" проведи контент-анализ видео и текста и укажи пол автора канала. Используй строго одно из фиксированных значений на русском языке: "женский" или "мужской". Если на видео присутствует дуэт/группа людей разного пола или однозначно определить пол невозможно, укажи "универсальный". Категорически запрещено писать любые другие слова, пояснения или сторонние описания в этом поле.

- ПРАВИЛО ЯЗЫКА ДЛЯ ОСТАЛЬНЫХ БЛОКОВ: Во всех смысловых блоках (ключах JSON), кроме полей "entities" и "brands", должен использоваться ИСКЛЮЧИТЕЛЬНО русский язык (включая все добавляемые синонимы). Сами ключи JSON должны оставаться на английском языке, как показано в шаблоне ниже. 

- ПРАВИЛО ОПРЕДЕЛЕНИЯ ФОРМАТА ВИДЕО (ПРИОРИТЕТНЫЙ СПИСОК): Для заполнения поля "video_format" сопоставь контент 3 прикрепленных видеороликов со следующим фиксированным списком форматов: 
  * GRWM (Get Ready With Me) 
  * Обзор 
  * Распаковка 
  * До / После 
  * Разговорный ролик 
  * Инструкции / Лайфхаки 
  * Скетч / Юмористический ролик 
  * Тест-драйв / Проверка на стойкость 
  * Примерка / Показ одежды 
  * АСМР (ASMR) 
  * Образ дня / Аутфит 
  Если видео строго соответствует формату из этого списка, укажи именно его название из списка (и добавь к нему близкие русскоязычные синонимы формата в этом же поле). 
  Если и только если ни один формат из списка абсолютно не подходит под контент видео, определи фактический формат в свободной форме на русском языке: укажи до 3 конкретных форматов по видео и подбери к ним близкие синонимичные названия. 

- Кросс-анализ видео (OCR + Визуал + Аудио): Внимательно анализируй видеоряд и аудиодорожку. Считывай текст на экране (OCR). Сопоставляй текстовые маркеры с экрана, звучащую речь и подписи к постам из JSON. Извлекай только высокосемантичные токены (бренды, категории товаров, действия). Полностью игнорируй визуальный шум и призывы к действию ("подпишись", "ссылка", "часть 1"). 

Шаблон вывода (Выведи ТОЛЬКО один валидный JSON-объект. Не оборачивай его в маркдаун-блоки json ... , не пиши никаких вводных слов, приветствий или комментариев от себя):
{{
  "blogger_gender": "значение_пола",
  "domain": "основная_ниша_блогера, тематика_канала, связанные_индустрии_и_синонимы",
  "keywords": "ключевые_слова, извлеченные_продукты_с_уточнением_зоны_применения, действия_в_кадре, темы_постов",
  "entities": "фактические_названия_локаций_городов_стран_и_имен_на_языке_оригинала",
  "brands": "фактические_названия_брендов_продуктов_сервисов_и_платформ_на_языке_оригинала,
  "audio_and_delivery": "характеристики голоса и звука на русском языке, манера подачи, добавленные аудио-синонимы",
  "video_format": "формат_из_списка_и_его_русские_синонимы",
  "visual_style": "характеристики_планов, освещения, цветокора, визуальные_маркеры",
  "vibe": "эмоциональная_подача, атмосфера_видео, общее_позиционирование"
}}
""".strip()

def analyze_one_profile(client, profile_dir: Path, overwrite: bool = False) -> dict:
    output_path = profile_dir / "gemini_analysis.json"

    if output_path.exists() and not overwrite:
        return {
            "status": "skipped_exists",
            "profile_dir": str(profile_dir),
            "output_path": str(output_path)
        }

    bio_text, profile_data = read_profile_files(profile_dir)
    video_paths = get_video_paths(profile_dir, limit=3)
    
    if len(video_paths) == 0:
        return {"status": "no video"}

    if not bio_text and not profile_data:
        raise RuntimeError(f"Нет bio.txt и profile.json: {profile_dir}")

    uploaded_files = []

    for video_path in video_paths:
        print(f"Загружаю видео: {video_path}")
        uploaded = client.files.upload(file=str(video_path))
        uploaded = wait_file_active(client, uploaded)
        uploaded_files.append(uploaded)

    prompt = build_prompt(profile_dir, bio_text, profile_data)

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            *uploaded_files,
            prompt,
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    result = extract_json(response.text)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return {
        "status": "ok",
        "profile_dir": str(profile_dir),
        "videos_sent": [str(p) for p in video_paths],
        "output_path": str(output_path)
    }


def analyze_all_profiles(
    root_dir: str | Path = ".",
    start: int = 1,
    end: int = 418,
    overwrite: bool = False,
    sleep_between_requests: float = 2.0,
) -> list[dict]:
    if not GEMINI_API_KEY:
        raise RuntimeError("Нет GEMINI_API_KEY в .env")

    client = genai.Client(api_key=GEMINI_API_KEY)

    root_dir = Path(root_dir)
    results = []

    for i in range(start, end + 1):
        number_dir = root_dir / str(i)
        profile_dir = find_profile_dir(number_dir)

        if profile_dir is None:
            item = {
                "number": i,
                "status": "no_profile_dir",
                "folder": str(number_dir)
            }
            print(item)
            results.append(item)
            continue

        print(f"\n=== {i}/{end}: {profile_dir} ===")

        try:
            item = analyze_one_profile(
                client=client,
                profile_dir=profile_dir,
                overwrite=overwrite
            )
            complete.append(i)
            item["number"] = i
            print(item)

        except Exception as e:
            item = {
                "number": i,
                "status": "error",
                "profile_dir": str(profile_dir),
                "error": str(e)
            }
            print(item)

            
        results.append(item)

        # Сохраняем общий прогресс после каждого профиля
        progress_path = root_dir / "gemini_batch_results.json"
        with open(progress_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        time.sleep(sleep_between_requests)

    return results

In [27]:
results = analyze_all_profiles(
    root_dir=".",      # если папки 1...418 лежат рядом со скриптом
    start=273,
    end=273,
    overwrite=True   # не пересоздавать, если gemini_analysis.json уже есть
)


=== 273/273: 273\1485873279 ===
Загружаю видео: 273\1485873279\videos\01_DYJ39Z_MGpC.mp4
Gemini обрабатывает файл: files/a5w2xuzdgf6b
Загружаю видео: 273\1485873279\videos\02_DYdCOk0smbB.mp4
Gemini обрабатывает файл: files/rxxz537fy0pm
Загружаю видео: 273\1485873279\videos\03_DXJcW-3jgAb.mp4
Gemini обрабатывает файл: files/gqlzq70j86hw
Gemini обрабатывает файл: files/gqlzq70j86hw
{'status': 'ok', 'profile_dir': '273\\1485873279', 'videos_sent': ['273\\1485873279\\videos\\01_DYJ39Z_MGpC.mp4', '273\\1485873279\\videos\\02_DYdCOk0smbB.mp4', '273\\1485873279\\videos\\03_DXJcW-3jgAb.mp4'], 'output_path': '273\\1485873279\\gemini_analysis.json', 'number': 273}


In [87]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer


MODEL_NAME = "BAAI/bge-m3"


TEMPLATE = """Профиль блогера (пол - {blogger_gender}) ориентирован на следующие направления и тематики: {domain}. 
Основные поисковые маркеры, теги и ключевые слова контента: {keywords}. 
В материалах и видеороликах фактически зафиксированы следующие географические локации и персоны: {entities}. 
Упомянутые и продемонстрированные коммерческие бренды, продукты, компании и платформы: {brands}. 
Основные форматы видеороликов, сценарные подходы и типы контента: {video_format}.
Визуальное оформление профиля, стиль съемки, свет и особенности монтажа: {visual_style}. 
Звуковое сопровождение, характеристики аудиодорожки, манера речи и подача автора в кадре: {audio_and_delivery}.
Общая атмосфера, позиционирование, вайб аккаунта и ценовой сегмент целевой аудитории: {vibe}.
"""


FIELDS = [
    "blogger_gender",
    "domain",
    "keywords",
    "entities",
    "brands",
    "video_format",
    "visual_style",
    "audio_and_delivery",
    "vibe",
]


def find_profile_dir(number_dir: Path) -> Path | None:
    """
    В папке 1, 2, 3... ищем единственную папку с id.
    Если папок несколько, берём ту, где есть gemini_analysis.json.
    """
    if not number_dir.exists() or not number_dir.is_dir():
        return None

    subdirs = [p for p in number_dir.iterdir() if p.is_dir()]

    if len(subdirs) == 1:
        return subdirs[0]

    for p in subdirs:
        if (p / "gemini_analysis.json").exists():
            return p

    return None


def clean_json_text(text: str) -> str:
    text = text.strip()

    # На случай если Gemini вернул ```json ... ```
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    return text.strip()


def parse_json_maybe_raw_response(path: Path) -> dict:
    """
    Поддерживает оба формата:

    1. Нормальный:
    {
      "domain": "...",
      ...
    }

    2. Сырой:
    {
      "raw_response": "{\\n  \\"domain\\": \\"...\\"\\n}"
    }
    """

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, dict) and "raw_response" in data:
    
        raw = data["raw_response"]

        if isinstance(raw, dict):
            return raw

        if isinstance(raw, str):
            raw = clean_json_text(raw)

            try:
                return json.loads(raw)
            except json.JSONDecodeError:
                # Последняя попытка: вытащить объект между первой { и последней }
                start = raw.find("{")
                end = raw.rfind("}")

                if start != -1 and end != -1 and end > start:
                    return json.loads(raw[start:end + 1])

                raise

    return data


def safe_value(data: dict, key: str) -> str:
    value = data.get(key, "")

    if value is None:
        return ""

    if isinstance(value, list):
        return ", ".join(map(str, value))

    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    return str(value).strip()


def render_blogger_text(analysis: dict) -> str:
    values = {
        field: safe_value(analysis, field)
        for field in FIELDS
    }

    return TEMPLATE.format(**values)


def load_profile_json(profile_dir: Path) -> dict:
    path = profile_dir / "profile.json"

    if not path.exists():
        return {}

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}


def has_nonempty_videos(number_dir: Path, profile_dir: Path | None = None) -> bool:
    """
    Проверяем, что есть папка videos и она не пустая.

    Поддерживает оба варианта структуры:
    1. number_dir/videos
    2. number_dir/profile_id/videos
    """
    candidates = [number_dir / "videos"]

    if profile_dir is not None:
        candidates.append(profile_dir / "videos")

    for videos_dir in candidates:
        if videos_dir.exists() and videos_dir.is_dir():
            try:
                if any(videos_dir.iterdir()):
                    return True
            except OSError:
                return False

    return False


def has_all_required_fields(data: dict) -> bool:
    """
    Проверяем, что в gemini_analysis.json есть все поля из FIELDS
    и они не пустые.
    """
    if not isinstance(data, dict):
        return False

    for field in FIELDS:
        if field not in data:
            return False

        if safe_value(data, field) == "":
            return False

    return True


def collect_blogger_texts(
    root_dir: str | Path = ".",
    start: int = 1,
    end: int = 120,
) -> pd.DataFrame:
    root_dir = Path(root_dir)
    rows = []

    for i in range(start, end + 1):
        number_dir = root_dir / str(i)

        if not number_dir.exists() or not number_dir.is_dir():
            continue

        profile_dir = find_profile_dir(number_dir)

        if profile_dir is None:
            continue

        # Скипаем, если нет videos или videos пустая
        if not has_nonempty_videos(number_dir, profile_dir):
            continue

        analysis_path = profile_dir / "gemini_analysis.json"

        # Скипаем, если нет gemini_analysis.json
        if not analysis_path.exists():
            continue

        try:
            analysis = parse_json_maybe_raw_response(analysis_path)

            # Скипаем, если нет всех нужных полей
            if not has_all_required_fields(analysis):
                continue

            profile = load_profile_json(profile_dir)

            rendered_text = render_blogger_text(analysis)

            row = {
                "folder_number": i,
                "profile_dir": str(profile_dir),
                "analysis_path": str(analysis_path),

                "profile_id": profile_dir.name,
                "username": profile.get("username"),
                "full_name": profile.get("full_name"),
                "followers": profile.get("followers"),

                "blogger_gender": safe_value(analysis, "blogger_gender"),
                "domain": safe_value(analysis, "domain"),
                "keywords": safe_value(analysis, "keywords"),
                "entities": safe_value(analysis, "entities"),
                "brands": safe_value(analysis, "brands"),
                "video_format": safe_value(analysis, "video_format"),
                "visual_style": safe_value(analysis, "visual_style"),
                "audio_and_delivery": safe_value(analysis, "audio_and_delivery"),
                "vibe": safe_value(analysis, "vibe"),

                "rendered_text": rendered_text,
            }

            rows.append(row)

        except Exception:
            # Важно: ничего не добавляем в rows
            continue

    return pd.DataFrame(rows)


def add_embeddings(
    df: pd.DataFrame,
    model_name: str = MODEL_NAME,
    text_column: str = "rendered_text",
    batch_size: int = 16,
) -> tuple[pd.DataFrame, np.ndarray]:
    valid_mask = df[text_column].fillna("").str.strip() != ""
    texts = df.loc[valid_mask, text_column].tolist()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Model: {model_name}")
    print(f"Texts to embed: {len(texts)}")

    model = SentenceTransformer(model_name, device=device)
    model.max_seq_length = 4096

    valid_embeddings = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    valid_embeddings = np.asarray(valid_embeddings, dtype=np.float32)

    embedding_dim = valid_embeddings.shape[1] if len(valid_embeddings) else 0
    all_embeddings = np.zeros((len(df), embedding_dim), dtype=np.float32)

    all_embeddings[valid_mask.to_numpy()] = valid_embeddings

    df = df.copy()
    df["embedding"] = list(all_embeddings)

    return df, all_embeddings
    

In [88]:
df = collect_blogger_texts(
        root_dir=".",
        start=1,
        end=418,
    )

In [89]:
df[df.folder_number == 273]

,folder_number,profile_dir,analysis_path,profile_id,username,full_name,followers,blogger_gender,domain,keywords,entities,brands,video_format,visual_style,audio_and_delivery,vibe,rendered_text
190,273,273\1485873279,273\1485873279\gemini_analysis.json,1485873279,kisaliza_7,Лиза Киса | lifestyle creator• Москва 🧿,1529,женский,"лайфстайл, кальянная культура, домашнее садово...","кальянный табак, курение кальяна, выдувание ды...","Москва, Лиза","Золотое Яблоко, Fix Price, Tellmi, YASOMA, Mua...","Инструкции / Лайфхаки, Обзор, АСМР (ASMR), Раз...","крупный план, средний план, домашняя обстановк...","разговорная речь, естественная интонация, спок...","уютный, домашний, расслабленный, меланхоличный...",Профиль блогера (пол - женский) ориентирован н...


In [90]:
print(f"Собрано строк: {len(df)}")
df

Собрано строк: 284


,folder_number,profile_dir,analysis_path,profile_id,username,full_name,followers,blogger_gender,domain,keywords,entities,brands,video_format,visual_style,audio_and_delivery,vibe,rendered_text
0,1,1\5569396957,1\5569396957\gemini_analysis.json,5569396957,gvardis.m,Маша Гвардис | обзоры одежды и beauty | ugc | WB,3780,женский,"бьюти, косметика, мода, стиль, обзоры одежды, ...","блеск для губ, двухсторонний стик для лица, ск...","Маша, Маша Гвардис","Letimi, Wildberries, WB, Dior, Hello Kitty, Be...","GRWM (Get Ready With Me), Примерка / Показ оде...","крупный план, средний план, динамичный монтаж,...","естественная речь, разговорный тон, дружелюбны...","атмосфера подружки, искренность, дружелюбное о...",Профиль блогера (пол - женский) ориентирован н...
1,4,4\58159981827,4\58159981827\gemini_analysis.json,58159981827,as_crimeaa,Анастасия|Beauty|Обзоры|Севастополь,10228,женский,"бьюти-индустрия, уход за кожей, обзоры космети...","сыворотка для лица, сыворотка с витамином С, л...","Севастополь, Крым, Sevastopol, Crimea","Wildberries, VT Cosmetics, Vei Skin, Nikk Mole...","Скетч / Юмористический ролик, Обзор, подборка ...","портретная съемка, дневной свет, природный фон...","естественный закадровый голос, динамичный монт...","эстетичный, дружелюбный, забавный, юмористичес...",Профиль блогера (пол - женский) ориентирован н...
2,5,5\53579029361,5\53579029361\gemini_analysis.json,53579029361,pon.alexa,Alexandra,13875,женский,"уход за волосами, бьюти-индустрия, красота, ух...","рост волос, фен для волос, термозащита для вол...","Moscow, South Africa, Москва, Саша, Alexandra","Fichi, fichibrand, Wildberries","Разговорный ролик, Инструкции / Лайфхаки, бьют...","крупный план, зеркало, селфи-видео, комнатное ...","спокойный голос, мягкий тембр, четкая дикция, ...","доверительный, эстетичный, экспертный, искренн...",Профиль блогера (пол - женский) ориентирован н...
3,7,7\3685356465,7\3685356465\gemini_analysis.json,3685356465,kate_mom2,КАТЯ | мама двоих | UGC creator,1793,женский,"бьюти, уход за кожей, создание UGC-контента, м...","замороженный огурец для лица, натуральный масс...",КАТЯ,"Dr. Charm, ditalir_cosmetic, Wildberries, Ozon...","Обзор, Инструкции / Лайфхаки, бьюти-туториал, ...","крупные планы лица, предметная макросъемка, ес...","динамичная музыка, фоновые биты, ритмичный мон...","эстетичный, расслабляющий, уютный, забота о се...",Профиль блогера (пол - женский) ориентирован н...
4,9,9\65128479754,9\65128479754\gemini_analysis.json,65128479754,aammmmm79,Makeup model | FREE PALESTINA 🇵🇸,1622,женский,"бьюти-индустрия, профессиональный макияж, моде...","подводка для глаз, жидкий хайлайтер для скул, ...","Alina, PALESTINA, Hadija Baimuradova, Fatima S...",Instagram,"Образ дня / Аутфит, демонстрация макияжа, бьют...","крупный план, кольцевая лампа, профессиональны...","фоновая музыка, без разговорной речи, динамичн...","элегантность, утонченность, женственность, бью...",Профиль блогера (пол - женский) ориентирован н...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
279,406,406\67232071868,406\67232071868\gemini_analysis.json,67232071868,_ayawaska__,Aya Dan 🎧🫧 бьюти как стиль жизни,38370,женский,"бьюти, уход за лицом, массаж лица, фитнес, здо...","скребок гуаша для лица, сыворотка для лица, кр...","Белград, Aya Dan, Aya, Снежана","Instagram, Telegram, syleimanova_hair, Perfect...","Инструкции / Лайфхаки, До / После, Разговорный...","дневной свет, крупный план лица, кадры из спор...","приятный голос, закадровый голос, динамичная м...","мотивирующий, вдохновляющий, эстетичный, друже...",Профиль блогера (пол - женский) ориентирован н...
280,407,407\2010092419,407\2010092419\gemini_analysis.json,2010092419,pavlowna,Aнастасия • рилсмейкер • ugc/influencer • бьюти,16660,женский,"бьюти, мода, уход, стиль, одежда, лайфстайл, к...","косметичка для хранения, лаковая сумка, хлопко...","Москва, Рождественка","AliExpress, Intimissimi, Lime, Birdy, Yougo, L...","Обзор, Примерка / Показ одежды, бьюти-туториал...","средний план

In [91]:
import torch; print('torch:', torch.__version__); print('cuda:', torch.cuda.is_available())

torch: 2.6.0+cu124
cuda: True


In [92]:
df, embeddings = add_embeddings(df)

Device: cuda
Model: BAAI/bge-m3
Texts to embed: 284


Batches: 100%|██████████| 18/18 [00:11<00:00,  1.51it/s]


In [34]:
df2[df2.folder_number == 273]

,folder_number,profile_dir,analysis_path,profile_id,username,full_name,followers,blogger_gender,domain,keywords,entities,brands,video_format,visual_style,audio_and_delivery,vibe,rendered_text,embedding


In [93]:
assert len(df) == len(embeddings), (
    f"Размеры не совпадают: dfpr={len(df)}, embeddings={len(embeddings)}"
)

# Добавляем эмбеддинги в dfpr
df2 = df.copy()
df2["embedding"] = list(embeddings)

# Сохраняем
df2.to_pickle("profiles_with_embeddings.pkl")

# Считываем обратно
# dfpr = pd.read_pickle("dfpr_with_embeddings.pkl")

# # Проверка
# dfpr.head()

In [154]:
df

,folder_number,profile_dir,analysis_path,profile_id,username,full_name,followers,blogger_gender,domain,keywords,entities,brands,video_format,visual_style,audio_and_delivery,vibe,rendered_text,embedding
0,1,1\5569396957,1\5569396957\gemini_analysis.json,5569396957,gvardis.m,Маша Гвардис | обзоры одежды и beauty | ugc | WB,3780,женский,"бьюти, косметика, мода, стиль, обзоры одежды, ...","блеск для губ, двухсторонний стик для лица, ск...","Маша, Маша Гвардис","Letimi, Wildberries, WB, Dior, Hello Kitty, Be...","GRWM (Get Ready With Me), Примерка / Показ оде...","крупный план, средний план, динамичный монтаж,...","естественная речь, разговорный тон, дружелюбны...","атмосфера подружки, искренность, дружелюбное о...",Профиль блогера (пол - женский) ориентирован н...,"[-0.05032675, 0.0066294554, -0.010729739, -0.0..."
1,4,4\58159981827,4\58159981827\gemini_analysis.json,58159981827,as_crimeaa,Анастасия|Beauty|Обзоры|Севастополь,10228,женский,"бьюти-индустрия, уход за кожей, обзоры космети...","сыворотка для лица, сыворотка с витамином С, л...","Севастополь, Крым, Sevastopol, Crimea","Wildberries, VT Cosmetics, Vei Skin, Nikk Mole...","Скетч / Юмористический ролик, Обзор, подборка ...","портретная съемка, дневной свет, природный фон...","естественный закадровый голос, динамичный монт...","эстетичный, дружелюбный, забавный, юмористичес...",Профиль блогера (пол - женский) ориентирован н...,"[-0.03905878, -0.0026017767, -0.015176701, -0...."
2,5,5\53579029361,5\53579029361\gemini_analysis.json,53579029361,pon.alexa,Alexandra,13875,женский,"уход за волосами, бьюти-индустрия, красота, ух...","рост волос, фен для волос, термозащита для вол...","Moscow, South Africa, Москва, Саша, Alexandra","Fichi, fichibrand, Wildberries","Разговорный ролик, Инструкции / Лайфхаки, бьют...","крупный план, зеркало, селфи-видео, комнатное ...","спокойный голос, мягкий тембр, четкая дикция, ...","доверительный, эстетичный, экспертный, искренн...",Профиль блогера (пол - женский) ориентирован н...,"[-0.044650707, -0.016069997, -0.031890113, -0...."
3,7,7\3685356465,7\3685356465\gemini_analysis.json,3685356465,kate_mom2,КАТЯ | мама двоих | UGC creator,1793,женский,"бьюти, уход за кожей, создание UGC-контента, м...","замороженный огурец для лица, натуральный масс...",КАТЯ,"Dr. Charm, ditalir_cosmetic, Wildberries, Ozon...","Обзор, Инструкции / Лайфхаки, бьюти-туториал, ...","крупные планы лица, предметная макросъемка, ес...","динамичная музыка, фоновые биты, ритмичный мон...","эстетичный, расслабляющий, уютный, забота о се...",Профиль блогера (пол - женский) ориентирован н...,"[-0.06327672, 0.006668137, -0.025280043, 0.008..."
4,9,9\65128479754,9\65128479754\gemini_analysis.json,65128479754,aammmmm79,Makeup model | FREE PALESTINA 🇵🇸,1622,женский,"бьюти-индустрия, профессиональный макияж, моде...","подводка для глаз, жидкий хайлайтер для скул, ...","Alina, PALESTINA, Hadija Baimuradova, Fatima S...",Instagram,"Образ дня / Аутфит, демонстрация макияжа, бьют...","крупный план, кольцевая лампа, профессиональны...","фоновая музыка, без разговорной речи, динамичн...","элегантность, утонченность, женственность, бью...",Профиль блогера (пол - женский) ориентирован н...,"[-0.027278384, -0.0047691353, -0.014732255, 0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,254,254\6834283387,254\6834283387\gemini_analysis.json,6834283387,dariiruse,Дарья | Контент-продюсер | Москва,861,женский,"производство бьюти-контента, контент-продюсиро...","гелевый карандаш для контура губ, тональный ти...","Москва, Сочи, Даша, Настя, Анастасия Шиндязова...","Lic, Ritter Sport","промо-ролик, презентация косметики, коммерческ...","студийный яркий свет, макро-планы, крупные кад...","динамичная музыка, фоновый электронный бит, пр...","эстетичный, профессиональный, стильный, вдохно...",Профиль блогера (пол - женский) ориентирован н...,"[-0.052611414, -0.00079536514, -0.02544897, -0..."
176,256,256\187375238,256\187375238\gemini_analysis.json,187375238,zotova.vasi

In [ ]:
# df.to_csv("profiles_v0.csv", index=False)

In [156]:
dfpr = pd.read_excel("Брифы.xlsx")
dfpr

,Описание рекламной кампании (Бриф для ИИ),Продукт / Услуга,Формат видео,Tone of Voice (ToV),Блогер,AI-Score
0,Хочу запустить кампанию под наш новый продукт....,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
248,NaN,NaN,NaN,NaN,NaN,NaN
249,NaN,NaN,NaN,NaN,NaN,NaN
250,NaN,NaN,NaN,NaN,NaN,NaN
251,NaN,NaN,NaN,NaN,NaN,NaN


In [157]:
dfpr = dfpr.drop(columns=["Блогер", "AI-Score"])

In [158]:
dfpr = dfpr.dropna()

In [159]:
df.drop(columns=["embedding"]).to_csv(
        "bloggers_texts_v3.csv",
        index=False,
        encoding="utf-8-sig",
    )

In [160]:
    # Parquet с embedding внутри df
df.to_parquet(
    "bloggers_texts_embeddings_v3.parquet",
    index=False,
)

In [161]:
np.save("bloggers_embeddings_v3.npy", embeddings)

In [162]:
df = pd.read_parquet("bloggers_texts_embeddings_v3.parquet")
blogger_embeddings = np.load("bloggers_embeddings_v3.npy")

In [163]:
CAMPAIGN_TEMPLATE = """Рекламируемый продукт или предоставляемая услуга в рамках кампании: {product}.
Требуемый формат видеопроизводства и тип интеграционного ролика: {video_format}.
Желаемая манера подачи контента, стиль коммуникации и Tone of Voice (ToV): {tone_of_voice}.
Полное описание рекламной кампании, цели интеграции, техническое задание и бриф для ИИ: {brief}."""

In [164]:
def safe_str(value) -> str:
    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return str(value).strip()


def render_campaign_text(row) -> str:
    return CAMPAIGN_TEMPLATE.format(
        product=safe_str(row["Продукт / Услуга"]),
        video_format=safe_str(row["Формат видео"]),
        tone_of_voice=safe_str(row["Tone of Voice (ToV)"]),
        brief=safe_str(row["Описание рекламной кампании (Бриф для ИИ)"]),
    )


dfpr = dfpr.copy()

dfpr["campaign_text"] = dfpr.apply(render_campaign_text, axis=1)

In [165]:
print(dfpr.loc[dfpr.index[0], "campaign_text"])

Рекламируемый продукт или предоставляемая услуга в рамках кампании: SPF-флюид для города.
Требуемый формат видеопроизводства и тип интеграционного ролика: GRWM (Get Ready With Me).
Желаемая манера подачи контента, стиль коммуникации и Tone of Voice (ToV): Вдохновляющий, спокойный, доверительный.
Полное описание рекламной кампании, цели интеграции, техническое задание и бриф для ИИ: Хочу запустить кампанию под наш новый продукт. Формат — эстетичная бьюти-рутина в формате утреннего ухода за лицом. Визуальный стиль: светлый интерьер, естественное освещение, минималистичный фон. Блогер показывает уход за кожей, наносит средство, транслируя повседневное вдохновение и эстетику ухода. Подача: спокойный голос, мягкий тембр, искренность и доверительный тон..


In [166]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer


MODEL_NAME = "BAAI/bge-m3"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 4096

campaign_embeddings = model.encode(
    dfpr["campaign_text"].tolist(),
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)

campaign_embeddings = np.asarray(campaign_embeddings, dtype=np.float32)

Batches: 100%|██████████| 3/3 [00:00<00:00,  4.49it/s]


In [169]:
dfpr

,Описание рекламной кампании (Бриф для ИИ),Продукт / Услуга,Формат видео,Tone of Voice (ToV),campaign_text
0,Хочу запустить кампанию под наш новый продукт....,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Рекламируемый продукт или предоставляемая услу...
6,Ищу блогеров для продвижения аптечной линейки....,Аптечная косметика с центеллой и ниацинамидом,Обзор,"Экспертный, доступный, дружелюбный",Рекламируемый продукт или предоставляемая услу...
12,Закупаю рекламу на наше премиальное средство -...,Парфюмированное масло для волос,АСМР (ASMR),"Эстетичный, премиальный, мягкий",Рекламируемый продукт или предоставляемая услу...
18,Мне нужны блогеры для продвижения средства ноч...,Ночная маска для губ,Разговорный ролик,"Уютный, интимный, комфортный",Рекламируемый продукт или предоставляемая услу...
24,Запускаем базовый уход: гель для умывания и кр...,Мужской гель для умывания и крем,Разговорный ролик,"Ироничный, позитивный, прямолинейный",Рекламируемый продукт или предоставляемая услу...
30,Реклама антицеллюлитного средства. Формат — мо...,Антицеллюлитный кофейный скраб,Инструкции / Лайфхаки,"Мотивирующий, энергичный, эмоциональный",Рекламируемый продукт или предоставляемая услу...
36,Ищу блогеров для продвижения уходового средств...,Лифтинг-сыворотка для лица,Инструкции / Лайфхаки,"Уверенный, поддерживающий, вдохновляющий",Рекламируемый продукт или предоставляемая услу...
42,"Нужно прорекламировать стойкое средство, перек...",Стойкий тональный крем и консилер,Тест-драйв / Проверка на стойкость,"Динамичный, позитивный, драйвовый",Рекламируемый продукт или предоставляемая услу...
48,Ищу блогеров под средство с эффектом объема. Н...,Масло-блеск для губ,До / После,"Легкий, эмоциональный, трендовый",Рекламируемый продукт или предоставляемая услу...
54,Продвигаем линейку ярких неоновых средств. Фор...,Неоновые кайалы (карандаши для глаз),Образ дня / Аутфит,"Креативный, эстетичный, художественный",Рекламируемый продукт или предоставляемая услу...


In [168]:
campaign_embeddings

array([[-0.06902512, -0.01316976,  0.01064422, ..., -0.03251183,
         0.00960061, -0.02807802],
       [-0.03884979, -0.01081775,  0.00404879, ..., -0.03322424,
         0.0069689 , -0.02851549],
       [-0.04292928, -0.01926232,  0.00918678, ..., -0.02997787,
        -0.00411329, -0.01510567],
       ...,
       [-0.06410065, -0.03964591,  0.00241893, ..., -0.00788057,
        -0.0126943 , -0.03389481],
       [-0.05238875, -0.00906219,  0.0042456 , ..., -0.06352897,
         0.00100866, -0.02069367],
       [-0.06256334, -0.01334662, -0.03623068, ..., -0.03499066,
        -0.01427551, -0.00603438]], shape=(43, 1024), dtype=float32)

In [174]:
assert len(dfpr) == len(campaign_embeddings), (
    f"Размеры не совпадают: dfpr={len(dfpr)}, embeddings={len(campaign_embeddings)}"
)

# Добавляем эмбеддинги в dfpr
dfpr = dfpr.copy()
dfpr["embedding"] = list(campaign_embeddings)

# Сохраняем
dfpr.to_pickle("dfpr_with_embeddings.pkl")

# # Считываем обратно
# dfpr = pd.read_pickle("dfpr_with_embeddings.pkl")

# # Проверка
# dfpr.head()

In [145]:
def get_relevant_and_irrelevant_bloggers_for_campaigns(
    df_campaigns: pd.DataFrame,
    df_bloggers: pd.DataFrame,
    campaign_embeddings: np.ndarray,
    blogger_embeddings: np.ndarray,
    top_relevant: int = 5,
    top_irrelevant: int = 2,
    zero_eps: float = 1e-8,
) -> pd.DataFrame:
    scores = campaign_embeddings @ blogger_embeddings.T

    results = []

    for campaign_pos, campaign_row in df_campaigns.reset_index().iterrows():
        campaign_scores = scores[campaign_pos]

        # Убираем абсолютный ноль релевантности
        valid_indices = np.where(np.abs(campaign_scores) > zero_eps)[0]

        if len(valid_indices) == 0:
            continue

        valid_scores = campaign_scores[valid_indices]

        # Самые релевантные среди ненулевых
        relevant_local_indices = np.argsort(-valid_scores)[:top_relevant]
        relevant_indices = valid_indices[relevant_local_indices]

        # Самые нерелевантные среди ненулевых
        irrelevant_local_indices = np.argsort(valid_scores)[:top_irrelevant]
        irrelevant_indices = valid_indices[irrelevant_local_indices]

        selected = []

        for rank, blogger_pos in enumerate(relevant_indices, start=1):
            selected.append({
                "match_type": "relevant",
                "rank": rank,
                "blogger_pos": blogger_pos,
                "score": float(campaign_scores[blogger_pos]),
            })

        for rank, blogger_pos in enumerate(irrelevant_indices, start=1):
            selected.append({
                "match_type": "irrelevant",
                "rank": rank,
                "blogger_pos": blogger_pos,
                "score": float(campaign_scores[blogger_pos]),
            })

        for item in selected:
            blogger_row = df_bloggers.iloc[item["blogger_pos"]]

            results.append({
                "campaign_original_index": campaign_row["index"],
                "campaign_name": safe_str(campaign_row.get("Продукт / Услуга")),
                "campaign_format": safe_str(campaign_row.get("Формат видео")),
                "campaign_tov": safe_str(campaign_row.get("Tone of Voice (ToV)")),
                "campaign_brief": safe_str(campaign_row.get("Описание рекламной кампании (Бриф для ИИ)")),

                "match_type": item["match_type"],
                "rank": item["rank"],
                "score": item["score"],

                "blogger_folder_number": blogger_row.get("folder_number"),
                "blogger_profile_id": blogger_row.get("profile_id"),
                "blogger_username": blogger_row.get("username"),
                "blogger_full_name": blogger_row.get("full_name"),
                "blogger_followers": blogger_row.get("followers"),
                "blogger_profile_dir": blogger_row.get("profile_dir"),

                "blogger_gender": blogger_row.get("blogger_gender"),
                "blogger_domain": blogger_row.get("domain"),
                "blogger_keywords": blogger_row.get("keywords"),
                "blogger_entities": blogger_row.get("entities"),
                "blogger_visual_style": blogger_row.get("visual_style"),
                "blogger_brands": blogger_row.get("brands"),
                "blogger_video_format": blogger_row.get("video_format"),
                "blogger_audio_and_delivery": blogger_row.get("audio_and_delivery"),
                "blogger_vibe": blogger_row.get("vibe"),
            })

    return pd.DataFrame(results)

In [146]:
len(dfpr)

43

In [147]:
ranking_df = get_relevant_and_irrelevant_bloggers_for_campaigns(
    df_campaigns=dfpr,
    df_bloggers=df,
    campaign_embeddings=campaign_embeddings,
    blogger_embeddings=blogger_embeddings,
    top_relevant=5,
    top_irrelevant=2,
)

ranking_df.to_csv(
    "campaign_to_bloggers_relevant_irrelevant_no_zero.csv",
    index=False,
    encoding="utf-8-sig",
)

ranking_df.to_excel(
    "campaign_to_bloggers_relevant_irrelevant_no_zero.xlsx",
    index=False,
)

ranking_df.head(30)

,campaign_original_index,campaign_name,campaign_format,campaign_tov,campaign_brief,match_type,rank,score,blogger_folder_number,blogger_profile_id,...,blogger_profile_dir,blogger_gender,blogger_domain,blogger_keywords,blogger_entities,blogger_visual_style,blogger_brands,blogger_video_format,blogger_audio_and_delivery,blogger_vibe
0,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,1,0.658765,16,62463700056,...,16\62463700056,женский,"уход за волосами, бьюти-лайфстайл, прически, у...","плетение кос, легкая прическа, пучок из волос,...","СПБ, Эмили","крупный план лица, средний план сзади, комнатн...",ARAVIA Laboratories,"Инструкции / Лайфхаки, GRWM (Get Ready With Me...","динамичная фоновая музыка, трендовые звуковые ...","уютная атмосфера, дружелюбный настрой, эстетич..."
1,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,2,0.657407,137,58146336414,...,137\58146336414,женский,"бьюти, уход за кожей, макияж, лайфстайл, стиль...","сыворотка для лица, крем для лица SPF, солевой...","СПБ, Питер, Севкабель, Света, Влад","крупный план, селфи-видео, съемка в зеркало, м...","Skinfood, bonyasbox, Naedine Studio, Elastica,...","GRWM, Инструкции / Лайфхаки, день из жизни, бь...","закадровый голос, спокойный тембр, мягкая пода...","эстетичный, расслабленный, романтичный, уютный..."
2,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,3,0.647581,251,5772898877,...,251\5772898877,женский,"мода, косметика, бьюти, уход за кожей, моделин...","коллагеновые патчи под глаза, крем для кожи во...",Москва,"крупный план, теплое домашнее освещение, демон...","Abib, Yasoma, Lunifera, VT Cosmetics","GRWM (Get Ready With Me), Распаковка, Обзор, у...","приятный женский голос, динамичная фоновая муз...","эстетичный, вдохновляющий, бьюти-атмосфера, за..."
3,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,4,0.641223,39,4013884722,...,39\4013884722,женский,"бьюти-блог, уход за кожей, UGC-контент, лайфст...","сыворотка для лица, спф-крем для лица, СС-крем...","Saint Petersburg, СПб, Наташа","крупный план, дневное освещение, эстетичные ка...","Tiam, Luxvisage, Essence, Beauty Bomb, Eva Mos...","GRWM, туториал по макияжу, лайфстайл влог, обз...","закадровый голос, спокойный тембр, фоновая муз...","уютный, легкий, эстетичный, расслабленный, лет..."
4,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,relevant,5,0.640058,202,415772084,...,202\415772084,женский,"лайфстайл, бьюти-индустрия, создание UGC конте...","утренняя рутина, тканевая маска для лица, увла...","Москва, Аня","эстетичный видеоряд, мягкий дневной свет, тепл...","Celimax, Boostlife, Reshape, Atmosphere, Skims...","GRWM (Get Ready With Me), повседневный влог, д...","приятный закадровый голос, спокойная манера ре...","уютная атмосфера, эстетичный минимализм, вдохн..."
5,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,irrelevant,1,0.529134,206,3401573950,...,206\3401573950,женский,"этнический стиль, татарская культура, национал...","татарский оберег ручной работы, серебряные дет...","Казань, Нариманова","теплый свет, крупные планы рук, детальная съем...","mizgel.jwl, tatarscarf, altynai_art_design","Скетч / Юмористический ролик, Инструкции / Лай...","динамичный бит, татарские народные мотивы, этн...","эстетичный, душевный, самобытный, ироничный, в..."
6,0,SPF-флюид для города,GRWM (Get Ready With Me),"Вдохновляющий, спокойный, доверительный",Хочу запустить кампанию под наш новый продукт....,irrelevant,2,0.536665,102,5912029361,...,102\59

In [148]:
import json
import pandas as pd
from pathlib import Path


def normalize_folder_number(value) -> str:
    """
    Приводим 1, 1.0, '1' к строке '1',
    чтобы совпало с ключами из insts.json.
    """
    if pd.isna(value):
        return ""

    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()


# Если файл называется insts.json
insts_path = Path("insts.json")

with open(insts_path, "r", encoding="utf-8") as f:
    insts = json.load(f)


ranking_df = ranking_df.copy()

ranking_df["blogger_folder_number_str"] = (
    ranking_df["blogger_folder_number"]
    .apply(normalize_folder_number)
)

ranking_df["blogger_instagram_url"] = (
    ranking_df["blogger_folder_number_str"]
    .map(insts)
)

# Можно убрать техническую колонку
ranking_df = ranking_df.drop(columns=["blogger_folder_number_str"])

ranking_df.to_excel(
    "campaign_to_bloggers_relevant_irrelevant_with_links.xlsx",
    index=False
)

ranking_df.to_csv(
    "campaign_to_bloggers_relevant_irrelevant_with_links.csv",
    index=False,
    encoding="utf-8-sig"
)